In [10]:
# -*- coding: utf-8 -*-
"""ocr_handwritten_maths_ready.ipynb"""

!pip install kagglehub transformers datasets evaluate jiwer accelerate sentencepiece pandas -q

import kagglehub
import os
import pandas as pd
from PIL import Image
from datasets import Dataset
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# 1. Download the IMAGE-BASED Math Dataset
print("Downloading PNG Handwritten Math dataset from Kaggle...")
DATASET_PATH = kagglehub.dataset_download("sumit17125/handwritten-images-dataset")
print(f"Dataset successfully downloaded to: {DATASET_PATH}")

# ==========================================
# 2. Exact Path Configuration
# ==========================================
LABELS_PATH = os.path.join(DATASET_PATH, "caption_data.csv")
IMAGES_DIR = os.path.join(DATASET_PATH, "Handwritten_equations", "Handwritten_equations", "Handwritten", "Dataset")

print(f"✅ Labels file targeted at: {LABELS_PATH}")
print(f"✅ Images folder targeted at: {IMAGES_DIR}\n")

# ==========================================
# 3. Parse the CSV ground truth file
# ==========================================
import pandas as pd
image_label_data = []

try:
    # Read the CSV file
    df = pd.read_csv(LABELS_PATH)
    print(f"CSV Columns found: {list(df.columns)}")

    # Dynamically grab the first two columns (Usually Image Name and Caption/Latex)
    img_col = df.columns[0]
    text_col = df.columns[1]

    for index, row in df.iterrows():
        filename = str(row[img_col]).strip()
        latex_text = str(row[text_col]).strip()

        # Skip empty labels
        if not latex_text or latex_text == 'nan':
            continue

        # Ensure the filename ends with .bmp since we know the folder contains bitmaps
        if not filename.endswith('.bmp'):
            filename += '.bmp'

        img_path = os.path.join(IMAGES_DIR, filename)

        # Only add if the image actually exists on disk
        if os.path.exists(img_path):
            image_label_data.append({
                "image": img_path,
                "text": latex_text
            })

    print(f"\n🎉 Valid samples found and paired: {len(image_label_data)}")
    if len(image_label_data) > 0:
        print("Sample data:", image_label_data[:2])
    else:
        print("⚠️ No images matched. The filenames in the CSV might not match the folder.")

except Exception as e:
     print(f"⚠️ Error reading the CSV file: {e}")
# 4. Create Hugging Face Dataset & Process
if len(image_label_data) > 0:
    dataset = Dataset.from_list(image_label_data)
    dataset = dataset.train_test_split(test_size=0.1)

    train_ds = dataset["train"]
    val_ds = dataset["test"]

    print(f"Train size: {len(train_ds)}")
    print(f"Val size: {len(val_ds)}")

    print("Loading TrOCR model and processor...")
    processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
    model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

    # Tell the model what token to start generating with
    model.config.decoder_start_token_id = processor.tokenizer.cls_token_id

# It is also good practice to ensure the pad token is explicitly set
    model.config.pad_token_id = processor.tokenizer.pad_token_id

    # Increase token limit for long math equations
    max_target_length = 128

    def transform(example_batch):
        images = [Image.open(x).convert("RGB") for x in example_batch["image"]]
        inputs = processor(images, return_tensors="pt")

        labels = processor.tokenizer(
            example_batch["text"],
            padding="max_length",
            max_length=max_target_length,
            truncation=True
        ).input_ids

        # Ensure padding tokens are ignored by the loss function (-100)
        labels = [[(l if l != processor.tokenizer.pad_token_id else -100) for l in label] for label in labels]

        inputs["labels"] = labels
        return inputs

    train_ds.set_transform(transform)
    val_ds.set_transform(transform)

    print("✅ Dataset transformed and ready for training!")

Using Colab cache for faster access to the 'handwritten-images-dataset' dataset.
Dataset successfully downloaded to: /kaggle/input/handwritten-images-dataset
✅ Labels file targeted at: /kaggle/input/handwritten-images-dataset/caption_data.csv
✅ Images folder targeted at: /kaggle/input/handwritten-images-dataset/Handwritten_equations/Handwritten_equations/Handwritten/Dataset

CSV Columns found: ['Column1', 'Column2']

🎉 Valid samples found and paired: 12167
Sample data: [{'image': '/kaggle/input/handwritten-images-dataset/Handwritten_equations/Handwritten_equations/Handwritten/Dataset/18_em_0.bmp', 'text': 'x _ { k } x x _ { k } + y _ { k } y x _ { k }'}, {'image': '/kaggle/input/handwritten-images-dataset/Handwritten_equations/Handwritten_equations/Handwritten/Dataset/18_em_10.bmp', 'text': '2 6'}]
Train size: 10950
Val size: 1217
Loading TrOCR model and processor...


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Dataset transformed and ready for training!


In [11]:
# ==========================================
# MODEL TRAINING LOOP
# ==========================================
import evaluate
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator

print("Setting up training configuration...")

# 1. Load Character Error Rate (CER) to evaluate our OCR's accuracy
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    # Replace the -100 padding tokens with the actual pad token so we can decode the text
    pred_ids[pred_ids == -100] = processor.tokenizer.pad_token_id
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id

    # Decode the predictions and actual labels back to human-readable strings
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_str = processor.batch_decode(labels_ids, skip_special_tokens=True)

    # Calculate Character Error Rate
    cer = cer_metric.compute(predictions=pred_str, references=labels_str)

    return {"cer": cer}

# 2. Define training hyperparameters
# 2. Define training hyperparameters
training_args = Seq2SeqTrainingArguments(
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    output_dir="./trocr-math-solver",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    fp16=True,
    learning_rate=4e-5,
    num_train_epochs=3,
    load_best_model_at_end=True,
    remove_unused_columns=False,    # <--- ADD THIS LINE HERE
)

# 3. Initialize the Trainer
trainer = Seq2SeqTrainer(
    model=model,
    processing_class=processor.tokenizer,  # <-- UPDATED HERE
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,
)

# 4. Start the engine!
print("🚀 Starting training loop. This will take some time...")
trainer.train()

# 5. Save the final tuned model to your Colab disk
trainer.save_model("./trocr-math-solver-final")
processor.save_pretrained("./trocr-math-solver-final")
print("🎉 Model fine-tuning complete and saved!")

Setting up training configuration...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0}.


🚀 Starting training loop. This will take some time...


Epoch,Training Loss,Validation Loss,Cer
1,0.443639,0.405265,0.395609
2,0.170387,0.169218,0.318929
3,0.063348,0.118124,0.307549


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['decoder.output_projection.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 Model fine-tuning complete and saved!


In [16]:
import torch
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# 1. Define the paths
# Make sure this matches the filename of the image you uploaded to Colab
IMAGE_PATH = "/content/chnage confim.JPG"
# This is where your Trainer saved the model
MODEL_PATH = "./trocr-math-solver-final"

print("Loading saved model and processor...")
# 2. Load the fine-tuned processor and model
processor = TrOCRProcessor.from_pretrained(MODEL_PATH)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_PATH)

# Move model to GPU if available for faster generation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 3. Load and prepare the image
print(f"Processing image: {IMAGE_PATH}")
image = Image.open(IMAGE_PATH).convert("RGB")
pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)

# 4. Generate the LaTeX text
print("Generating prediction...")
# We use max_new_tokens to give the model plenty of room for long equations
generated_ids = model.generate(pixel_values, max_new_tokens=128)

# 5. Decode the output
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("\n=== ✨ PREDICTION ✨ ===")
print(generated_text)
print("========================")

Loading saved model and processor...


Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

Processing image: /content/chnage confim.JPG
Generating prediction...

=== ✨ PREDICTION ✨ ===
3 x - y = 7


In [17]:
# Compress the folder into a single zip file
!zip -r trocr-model.zip ./trocr-math-solver-final

  adding: trocr-math-solver-final/ (stored 0%)
  adding: trocr-math-solver-final/tokenizer_config.json (deflated 50%)
  adding: trocr-math-solver-final/config.json (deflated 74%)
  adding: trocr-math-solver-final/processor_config.json (deflated 52%)
  adding: trocr-math-solver-final/model.safetensors (deflated 7%)
  adding: trocr-math-solver-final/tokenizer.json (deflated 82%)
  adding: trocr-math-solver-final/training_args.bin (deflated 53%)
  adding: trocr-math-solver-final/generation_config.json (deflated 60%)


In [18]:
from google.colab import files
files.download('trocr-model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>